In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
json_path = "/home/lrm/workspace/segment_net/logs/rellis3d_maxxvitv2_query_transformer_20250802-185128/history.json"

with open(json_path, 'r') as f:
    data = json.load(f)

results_dir = "./results_cm"
os.makedirs(results_dir, exist_ok=True)

def process_confmat(confmat_data):
    shape = confmat_data['shape']
    flat_data = confmat_data['data']
    cm = np.array(flat_data).reshape(shape)

    row_sums = cm.sum(axis=1, keepdims=True)
    cm_normalized = np.divide(cm, row_sums, where=row_sums!=0)

    return cm_normalized

def plot_confusion_matrix(cm, labels, title, filename):
    fig_size = (10, 8)
    font_size = 14

    plt.figure(figsize=fig_size)
    
    annot = np.where(cm >= 0.04, np.char.mod('%.2f', cm), '')

    ax = sns.heatmap(cm, annot=annot,
                     cmap='YlOrRd',
                     xticklabels=labels,
                     yticklabels=labels,
                     fmt="",
                     cbar=True,
                     vmin=0, vmax=1,
                     annot_kws={"fontsize": font_size - 2})

    plt.xlabel('Predicted', fontsize=18, fontweight='bold')
    plt.ylabel('True', fontsize=18, fontweight='bold')
    plt.gca().xaxis.set_label_coords(0.5, 1.05)
    plt.gca().yaxis.set_label_coords(-0.07, 0.5)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=font_size)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=font_size)

    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, filename))
    plt.show()
    plt.clf()



In [ ]:

sample_cm = data['test']['image']['confmat'][0]
num_classes = sample_cm['shape'][0]
lat_labels = list(range(num_classes))

modes = {
    'train': data.get('train', {}).get('confmat', []),
    'val': data.get('val', {}).get('confmat', []),
    'test': data.get('test', {}).get('image', {}).get('confmat', [])
}

for mode, confmat_list in modes.items():
    if confmat_list:
        cm_data = confmat_list[0]  
        cm_norm = process_confmat(cm_data)

        classes_to_keep = list(range(2, cm_norm.shape[0]))  
        cm_norm = cm_norm[np.ix_(classes_to_keep, classes_to_keep)]
        lat_labels_filtered = classes_to_keep

        plot_confusion_matrix(cm_norm, lat_labels_filtered, f"{mode.upper()} - Confusion Matrix", f"{mode}_cm.pdf")
        print(f"{mode.upper()} matrix saved at: {os.path.join(results_dir, f'{mode}_cm.pdf')}")
    else:
        if mode == "test":
            print("\n⚠ No confusion matrix found for TEST mode.")
            print("To generate it, please run the pipeline in TEST mode using a configuration file.")
            print("Example:")
            print("    python run.py --cfg cfg/rellis3d_dev.ini")
            print("Make sure that inside the .ini file you set:")
            print("    mode = test\n")
        else:
            print(f"No matrix found for '{mode}'.")


No matrix found 'train'.
No matrix found 'val'.
TEST matrix saved at: ./results_cm/test_cm.pdf


<Figure size 1000x800 with 0 Axes>